In [ ]:
import json

import Run_Metrics

# ---------------------------------------------------------------------------
# Closing task of the run (spec 011, T-05): it consolidates the bronze
# summary, the freshness of the web export and the pending landing files into
# one row of <catalog>.ops.run_metrics, and fails when a check is false.
# It only coordinates inputs and outputs: the rules live in Run_Metrics.
#
# The status written in the row is the state of this closing (the four
# checks), not the aggregate of the tasks: per-task state and durations live
# in system.lakeflow and this task never waits for them.
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "landing")
dbutils.widgets.text("export_prefix", "gold/web_export")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("started_at", "")


def _widget(name, default=""):
    """Widget value, or its default when the job did not pass it."""
    return dbutils.widgets.get(name) or default


CATALOG = _widget("catalog", "job_offers")
STORAGE_ACCOUNT = _widget("storage_account")
CONTAINER = _widget("container", "landing")
EXPORT_PREFIX = _widget("export_prefix", "gold/web_export")
RUN_ID = _widget("run_id")
JOB_ID = _widget("job_id")
STARTED_AT = _widget("started_at")

# Without these four the run cannot be identified nor read; without run_id
# there is no row (a fallback id would update another run's row).
MISSING = [
    name
    for name, value in (
        ("storage_account", STORAGE_ACCOUNT),
        ("run_id", RUN_ID),
        ("job_id", JOB_ID),
        ("started_at", STARTED_AT),
    )
    if not value
]
if MISSING:
    raise ValueError(
        "Faltan parametros obligatorios de Publish_Run_Summary: "
        + ", ".join(MISSING)
        + "."
    )

STORAGE_BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
print(f"Catalogo {CATALOG}, landing {STORAGE_BASE}, run {RUN_ID}")

In [ ]:
# The bronze summary is published by Bronze_Ingestion as a JSON task value
# (Bronze_Summary.to_json). taskValues.get("bronze_summary", ...) is not
# enough: the API needs the task that published it, and outside a job the
# call raises. The error is not propagated here: the 'bronze_summary' check
# is the one that reports it.
try:
    raw_summary = dbutils.jobs.taskValues.get(
        "Bronze_Ingestion", "bronze_summary", default=None
    )
except Exception as exc:
    print(f"Sin task value 'bronze_summary' del bronce: {exc}")
    raw_summary = None

bronze_summary = None
if raw_summary:
    try:
        bronze_summary = (
            json.loads(raw_summary)
            if isinstance(raw_summary, str)
            else raw_summary
        )
    except (TypeError, ValueError) as exc:
        print(f"El resumen del bronce no es JSON valido: {exc}")

if not isinstance(bronze_summary, dict):
    bronze_summary = None

In [ ]:
# data_date of the web export: the same value the dashboard validates, read
# as text (no Parquet download and no table query). parse_data_date never
# raises: a corrupt meta.json is published as null and the freshness check is
# the one that fails, so the row is written anyway.
META_PATH = f"{STORAGE_BASE}/{EXPORT_PREFIX}/meta.json"
raw_data_date = None
try:
    meta = json.loads(dbutils.fs.head(META_PATH, 65536))
    raw_data_date = meta.get("data_date")
except Exception as exc:
    print(f"No se pudo leer {META_PATH}: {exc}")

data_date = Run_Metrics.parse_data_date(raw_data_date)
print(f"data_date del export: {data_date}")

In [ ]:
# Metadata listing only: not a single Parquet is read. The landing is listed
# with the DataFrame API, which uses the credentials of the job (the Java
# Hadoop filesystem of the notebook does not have them in ABFS), and the
# last commit of each checkpoint with dbutils.fs.ls. A failed landing
# listing is reported as listing_ok=False, with its cause published in the
# checks, instead of passing as "nothing pending".
import datetime as dt


def _last_commit(source):
    """Last checkpoint commit of a source, and the listing error if any.

    Returns ``(mtime_ms, error)``: ``mtime_ms`` is the mtime of the newest
    commit in ms, or ``None`` when there is none; ``error`` is ``None`` when
    the directory could be listed and a short text otherwise.

    It is read when the run closes: a run that ingests no row writes no new
    commit (measured with Spark 4.2 locally), so the files that were already
    waiting keep counting as pending and the backlog is not hidden. The
    error is returned, not raised: a missing commits directory is the first
    run of the source, and every file older than the start counts then.
    """
    path = f"{CHECKPOINTS_ROOT}/{source}/ckpt/commits"
    try:
        entries = dbutils.fs.ls(path)
    except Exception as exc:
        print(f"Sin commits en {path}: {exc}")
        return None, f"commits {source}: {' '.join(str(exc).split())}"
    times = [
        int(entry.modificationTime)
        for entry in entries
        if getattr(entry, "modificationTime", None) is not None
    ]
    return (max(times) if times else None), None


def _epoch_ms(moment):
    """Epoch milliseconds of a modificationTime, pinned to UTC.

    The value must not move with the session zone: unix_timestamp and cast
    follow spark.sql.session.timeZone, so the datetime is pinned instead
    with dt.replace(tzinfo=dt.timezone.utc) before asking for its timestamp.
    """
    if moment is None:
        return None
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=dt.timezone.utc)
    return int(moment.timestamp() * 1000)


def _landing_files(source):
    """Metadata and listing status of the landing files of ``source``.

    binaryFile lists the directory recursively with the DataFrame API and
    only path and modificationTime are selected: the content column is never
    requested, so no file is read. Returns ``(files, error)``: ``error`` is
    None when the directory could be listed, so a broken listing never looks
    like "nothing pending" and its cause travels to the checks.
    """
    try:
        listed = (
            spark.read.format("binaryFile")
            .option("recursiveFileLookup", "true")
            .load(f"{STORAGE_BASE}/{source}/")
            .select("path", "modificationTime")
            .collect()
        )
    except Exception as exc:
        print(f"No se pudo listar {STORAGE_BASE}/{source}/: {exc}")
        return [], f"{source}: {' '.join(str(exc).split())}"
    found = [
        {
            "source": source,
            "path": entry["path"],
            "modification_ms": _epoch_ms(entry["modificationTime"]),
        }
        for entry in listed
        if entry["path"] and entry["modificationTime"] is not None
    ]
    return found, None


# The checkpoint root travels in the bronze summary (dev and prod use
# different prefixes); the one of the job is the fallback.
_summary_config = (bronze_summary or {}).get("config")
_summary_config = _summary_config if isinstance(_summary_config, dict) else {}
_raw_root = _summary_config.get("checkpoints_root")
CHECKPOINTS_ROOT = (
    _raw_root
    if isinstance(_raw_root, str) and _raw_root
    else f"{STORAGE_BASE}/_checkpoints"
)

files = []
listing_errors = []
listing_ok = True
for source in Run_Metrics.EXPECTED_SOURCES:
    source_files, source_error = _landing_files(source)
    files.extend(source_files)
    listing_ok = listing_ok and not source_error
    if source_error:
        listing_errors.append(source_error)

commits = {}
for source in Run_Metrics.EXPECTED_SOURCES:
    commit_ms, commit_error = _last_commit(source)
    commits[source] = commit_ms
    if commit_error:
        listing_errors.append(commit_error)
listing_error = "; ".join(listing_errors)

started = Run_Metrics._as_utc(STARTED_AT, "started_at")
run_start_ms = int(started.timestamp() * 1000)
print(
    f"Inicio del run: {started.isoformat()} ({run_start_ms} ms); "
    f"ficheros listados: {len(files)}; listado ok: {listing_ok}"
)
print(f"Commits de los checkpoints: {commits}")
if listing_error:
    print(f"Error del listado: {listing_error}")

In [ ]:
pending = Run_Metrics.pending_files(files, commits, run_start_ms)
checks = Run_Metrics.build_checks(
    bronze_summary=bronze_summary,
    data_date=data_date,
    pending=pending,
    listing_ok=listing_ok,
    listing_error=listing_error,
)

In [ ]:
# The status of the row is the state of this closing (the four checks), not
# the aggregate of the tasks: the per-task state lives in system.lakeflow.
failed = Run_Metrics.failed_checks(checks)
row = Run_Metrics.build_run_row(
    run_id=str(RUN_ID),
    job_id=str(JOB_ID),
    started_at=STARTED_AT,
    bronze_summary=bronze_summary,
    data_date=data_date,
    status=("success" if not failed else "failed"),
    checks=checks,
)


class _PrintLogger:
    """Writer logger: a failed row must stay in the task output."""

    def warning(self, message):
        print(message)


# The write is best effort (Run_Metrics.record never propagates) and it is
# attempted always, also when a business task failed.
result = Run_Metrics.record(
    spark,
    Run_Metrics.table_name(CATALOG),
    row,
    logger=_PrintLogger(),
)

In [ ]:
print("Comprobaciones del run:")
print(json.dumps(checks, indent=2, sort_keys=True, ensure_ascii=False))
print(
    f"Escritura en {Run_Metrics.table_name(CATALOG)}: "
    + json.dumps(result, sort_keys=True)
)
print(f"run_id: {RUN_ID}")

In [ ]:
# The closing fails after writing the row: this task is the one that turns
# a false check into a red run.
problems = list(failed)
if result.get("status") != "ok":
    problems.append("record")
if problems:
    raise RuntimeError(
        "Comprobaciones del run en falso: "
        + ", ".join(problems)
        + ". Detalle: "
        + json.dumps(checks, sort_keys=True)
    )

dbutils.notebook.exit(
    json.dumps(
        {"run_id": RUN_ID, "checks": checks, "record": result},
        sort_keys=True,
    )
)